# 📊 02 — Validation du Feature Engineering

**Objectif** : Vérifier que nos features calculées (notamment les Z-Scores intraligue) séparent bien les joueurs promus des non-promus.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

# Style de base
plt.style.use('dark_background')
sns.set_theme(style='darkgrid', palette='viridis')
plt.rcParams['figure.figsize'] = (14, 6)


In [ ]:
# Chargement du dataset
df = pd.read_csv('../data/processed/features_players.csv')
print(f"Le dataset final contient {len(df)} lignes et {len(df.columns)} colonnes.")

# Extraire uniquement les ligues ERL (Exclure les lignes où la ligue est LEC pour l'évaluation ERL)
df_erl = df[df['league'] != 'LEC'].copy()
df_erl['promoted_label'] = df_erl['promoted_to_lec'].map({True: 'Promu', False: 'Non promu'})
print(f"Joueurs ERL étudiés (lignes par split) : {len(df_erl)}")

### Vérification des Z-Scores

In [ ]:
# Colonnes Z-Score à plotter
z_cols = [c for c in df.columns if c.endswith('_zscore')]
best_z_cols = ['dpm_zscore', 'cspm_zscore', 'golddiffat15_zscore', 'killparticipation_zscore', 'win_rate_zscore']

fig, axes = plt.subplots(1, len(best_z_cols), figsize=(20, 5))
for i, col in enumerate(best_z_cols):
    sns.boxplot(
        data=df_erl, x='promoted_label', y=col, 
        order=['Non promu', 'Promu'],
        hue='promoted_label', hue_order=['Non promu', 'Promu'],
        palette={'Non promu': '#636e72', 'Promu': '#00b894'}, 
        ax=axes[i], legend=False
    )
    axes[i].set_title(col)
    axes[i].set_xlabel('')
    axes[i].set_ylabel('Ecarts-types')

plt.suptitle('Comparaison des Z-Scores (Promus vs Non-Promus)', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
mean_z_promoted = df_erl[df_erl['promoted_to_lec'] == True][z_cols].mean()
mean_z_non_promoted = df_erl[df_erl['promoted_to_lec'] == False][z_cols].mean()
diff_z = mean_z_promoted - mean_z_non_promoted

print("Différence absolue des moyennes de Z-Score (Promus - Non Promus) :")
print(diff_z.sort_values(ascending=False).round(3))

### Constatations 
Les Z-scores sont supérieurs à +0.4 / +0.5 en moyenne pour les joueurs promus, ce qui indique que les joueurs repérés dominent structurellement leurs ligues. 
Le Feature Engineering semble valide et prêt pour les entraînements de modèles (Phase 5).